# 06 - Conditional scenario projection (Phase 6)

## What this notebook is for

Deliverable (2) of the practicum: **future hotspot scenario projection**. Two tracks,
both of which must be validated before anything is written to disk.

| Track | Question | Method | Validated by |
|---|---|---|---|
| **A** | Where is it hot, given the drivers? | Random forest regression of LST on NDVI, NDBI, built fraction, elevation, distance to coast, population density and LCZ class | RMSE and R2 on **spatially blocked** held-out data |
| **B** | Where will the land cover be? | CA-Markov on Dynamic World, in Python here and in MOLUSCE in QGIS | Kappa, its persistence baseline, the Pontius quantity/allocation split, and the figure of merit, against a **held-out year** |

Track B's projected land cover supplies Track A's projected predictors. Push those
through Track A's forest and you get a projected LST surface, under two scenarios -
business as usual and greening - and the difference between them.

## This notebook runs in THREE PARTS

| Part | Needs | What it does |
|---|---|---|
| **1** | Earth Engine | Probes the server-side classifier, probes Dynamic World coverage, and submits every export |
| **2** | Nothing but Python | Fits, validates and projects, entirely from the downloaded files |
| **3** | QGIS (optional) + Earth Engine | Reconciles MOLUSCE against the Python CA, then submits the **guarded** projection exports |

**Part 3's MOLUSCE step is optional.** If you skip it, the Python CA-Markov result
stands on its own and Phase 6 signs off without QGIS. Nothing is blocked.

## Four decisions taken before writing this, recorded here

| # | Decision | Why |
|---|---|---|
| 1 | The forest is fitted on **pixels at 100 m**, not on the 557 GN polygons | The deliverable is a projected *map*. A model fitted across 557 polygons is a property of that aggregation (`caveats.zonal_not_pixel`), and 557 rows cannot support a blocked split. 100 m is also `trends.fit_scale_m` and `spatial_stats.epoch_scale_m`, so the projection lands on the same grid as every Phase 4 and 5 product. |
| 2 | The CA runs at **the same 100 m grid** | The projected land cover supplies the projected predictors, and those go into a forest fitted at 100 m. Running the CA at 30 m and then applying the 100 m model would be applying a model at a scale it was never fitted at - a MAUP error wearing the look of extra detail. |
| 3 | Priority zones come from a **Phase-5 interim proxy**, not Phase 7's MCDA | Phase 7 has not been built. `prediction.priority_zones.source` is `phase5_interim` so the swap is findable, and `apply_greening_scenario` takes a plain zone list, so Phase 7 replaces the input without touching the scenario code. |
| 4 | Dynamic World **2018 -> 2021 calibrates, 2024 validates**; then 2018 -> 2024 projects | Roughly equal steps, and it keeps calibration and validation apart. Phase 5 probed coverage for 2016-2020 and 2024 but never 2021, so Step 2 re-probes before anything is calibrated. |

## Non-negotiable caveats

1. **This is not a forecast.** Every product here is a conditional scenario projection:
   *if* the calibrated transition rates continued, and *if* the fitted LST-driver
   relationship held. `prediction.require_validated` refuses to export or plot anything
   whose validation metrics were never computed.
2. **It is Land Surface Temperature, not air temperature.** Surface UHI can be roughly
   twice canopy-air UHI.
3. **The forest has never seen time.** It is fitted on one epoch's cross-section and
   then fed projected drivers - a space-for-time substitution.
4. **A random forest cannot extrapolate.** Projected pixels outside the training
   envelope are pinned to the edge of what it has seen. Step 11 counts them, and the
   guard fails the product above `prediction.rf.extrapolation.tolerance_fraction`.
5. **The training LST comes from L8+L9 only** (`landsat_oli_dry`). The Collection 2
   inter-calibration was tested over Colombo and failed; a model trained on a pooled
   series would learn the L7->L8 step as if it were geography.

---
# PART 1 - probe and export (needs Earth Engine)

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00-05 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("scenario_not_forecast", "lst_not_air_temp", "single_overpass",
             "valid_obs_required", "zonal_not_pixel"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - imports, staleness guard, constants

Phase 6 adds a whole module. If the notebook runs against a checkout that predates it,
every later cell fails with `AttributeError` at a different place, and the real cause -
*local changes committed but not pushed* - is three screens up. So this cell names every
function Phase 6 introduces and refuses loudly if any is absent.

It also carries **every import Parts 2 and 3 need**. Part 2 is meant to be runnable as a
separate session with no Earth Engine at all, so it must not depend on an import that
happens to live in a Part 1 cell.

In [ ]:
# COLAB: RUN THIS CELL
# Every import Parts 1, 2 AND 3 need, in one place.
import glob
import json
import time
import warnings

import ee
import numpy as np
import pandas as pd
from IPython.display import Image, display

from colombo_uhi import aoi, exports, landcover, prediction, spatial_stats, viz

_required = {
    "prediction": [
        # resolvers
        "resolve_predictors", "resolve_categorical", "resolve_rf_settings",
        "resolve_split", "resolve_ca_classes", "resolve_scenario",
        "resolve_projection_steps",
        # run-1 fixes
        "work_region", "resolve_class_grouping", "resolve_grouped_classes",
        "group_classes", "scoring_mask", "validate_projection",
        "ghsl_built_comparison", "read_priority_geometry",
        # run-2 fixes
        "resolve_scheme", "resolve_validation_intervals", "require_expected_area",
        # blocked splitting
        "spatial_block_ids", "require_enough_blocks", "blocked_split",
        "blocked_kfold", "random_row_split",
        # metrics
        "rmse", "r_squared", "confusion_matrix", "cohen_kappa",
        "persistence_baseline_kappa", "quantity_allocation_disagreement",
        "figure_of_merit",
        # Markov + CA
        "transition_matrix", "transition_probabilities", "markov_project",
        "projected_class_areas", "neighbourhood_potential", "ca_allocate",
        "ca_markov_project",
        # scenarios
        "interim_priority_zones", "apply_greening_scenario",
        "class_conditional_predictors", "paint_class_predictors",
        # the guard
        "extrapolation_flags", "build_validation_report", "require_validated",
        "validation_caption", "ValidationMissing",
        # scikit-learn side
        "fit_sklearn_rf", "score_rows", "blocked_cv_scores",
        "compare_split_strategies", "permutation_importance_frame",
        "compare_rf_implementations",
        # Earth Engine side
        "prediction_stack", "training_sample_selectors",
        "training_sample_collection", "export_training_sample",
        "read_training_sample", "fit_ee_rf", "ee_rf_explain",
        "project_lst_image", "export_projection", "predictor_band_order",
        "export_predictor_raster", "read_predictor_raster",
        "raster_sample_frame", "predict_surface", "lulc_class_image",
        "export_lulc_raster", "read_lulc_raster", "write_lulc_projection",
        "ghsl_built_cross_check",
    ],
    "viz": [
        "projection_caption", "landcover_palette",
        "plot_observed_vs_predicted", "plot_feature_importance",
        "plot_transition_matrix", "plot_lulc_validation",
        "plot_projected_lst", "plot_scenario_difference",
    ],
    "spatial_stats": [
        "covariate_stack", "dynamic_world_coverage", "read_zone_geodataframe",
        "export_division_geojson",
    ],
    "exports": ["export_name", "table_to_drive", "image_to_drive", "describe_tasks"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  prediction loaded from: {prediction.__file__}\n"
        "Fix, in order:\n"
        "  1. MOST LIKELY: local changes are COMMITTED BUT NOT PUSHED. This\n"
        "     notebook runs against the pushed repo. Check the HEAD line from\n"
        "     the clone cell, then git push and re-run FROM THE CLONE CELL.\n"
        "  2. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  3. If you uploaded this .ipynb by hand rather than opening it from\n"
        "     the repo, the notebook and src/ can be at DIFFERENT revisions."
    )
# .get() all the way down, so a checkout that predates Phase 6 entirely gets this
# message rather than a bare KeyError from the lookup itself.
_pred = params.get("prediction") or {}
for _path in ("rf.predictors", "split.method", "ca_markov.classes",
              "scenarios.greening", "priority_zones.top_n", "palettes.dynamic_world"):
    _node = _pred
    for _part in _path.split("."):
        _node = (_node or {}).get(_part) if isinstance(_node, dict) else None
    if _node is None:
        raise RuntimeError(
            f"config/params.yaml has no prediction.{_path} - your checkout "
            "predates Phase 6. Re-run the CLONE cell."
        )
print("PASS: all Phase 6 functions and params are present.")


# Run an interactive Earth Engine call; degrade to a note on a memory error.
# Everything that matters goes out through batch Export tasks, so an interactive
# call failing costs a convenience, never a result.
def _try_ee(label, call, default=None):
    try:
        return call()
    except ee.EEException as error:
        if "memory" not in str(error).lower():
            raise
        print(f"SKIPPED ({label}): exceeded the interactive memory limit.")
        print("  Not a failure - the batch exports carry the real products.")
        return default


RF = prediction.resolve_rf_settings(params)
SPLIT = prediction.resolve_split(params)
CA = params["prediction"]["ca_markov"]
CLASSES = prediction.resolve_ca_classes(params)
GROUPED_CLASSES = prediction.resolve_grouped_classes(params)
SCENARIOS = list(params["prediction"]["scenarios"])
HORIZONS = prediction.resolve_projection_steps(params)

RESPONSE = RF["response"]
PREDICTORS = RF["predictors"]
EPOCH = RF["epoch"]
SOURCE = RF["source"]
MODEL_SCALE = RF["scale_m"]
CA_SCALE = int(CA["raster_scale_m"])
CALIBRATION = [int(y) for y in CA["calibration_years"]]
VALIDATION_YEAR = int(CA["validation_year"])
BASE_YEARS = [int(y) for y in CA["projection_base_years"]]
INTERVALS = prediction.resolve_validation_intervals(params)
LULC_YEARS = sorted({*BASE_YEARS, *(y for triplet in INTERVALS for y in triplet)})

os.makedirs("data/outputs", exist_ok=True)
os.makedirs("data/interim", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# ---- THE REGION GUARD (added after Colab run 1) ---------------------------
# Run 1 used aoi.analysis_region - Western Province buffered by the 25 km SUHII
# ring - as the ANALYSIS unit. That is a bounding box for masks and composites.
# It measured 18,090 km2 against the district's 699, and every number the run
# produced described a regional elevation gradient and a lot of Indian Ocean.
# One getInfo, once, catches the entire class of error.
WORK_REGION = prediction.work_region(params)
_area = _try_ee("region area", lambda: aoi.area_km2(WORK_REGION).getInfo())
_expected = float(params["aoi"]["expected_areas_km2"]["district"])
if _area is None:
    print("*** Could not measure the region. Do NOT proceed - the whole point")
    print("*** of this guard is that an over-large region looks like success.")
elif abs(_area - _expected) / _expected > 0.20:
    raise RuntimeError(
        f"the work region measures {_area:,.0f} km2, but Colombo District is "
        f"~{_expected:,.0f} km2.\n"
        "MOST LIKELY: something is using aoi.analysis_region (Western Province "
        "+ a 25 km ring, ~18,000 km2) as the analysis unit. That is a bounding "
        "box for masks and composites, not a study area. Colab run 1 did "
        "exactly this and had to be discarded.\n"
        "prediction.work_region() is the analysis unit; notebooks 04 and 05 use "
        "aoi.colombo_district(params).geometry() for the same thing."
    )
else:
    print(f"PASS: work region {_area:,.1f} km2, within 20 % of the expected "
          f"{_expected:,.0f} km2 for Colombo District.")

print()
print("Response   :", RESPONSE, "from", SOURCE, "epoch", EPOCH, f"at {MODEL_SCALE} m")
print("Predictors :", PREDICTORS, "| categorical:", RF["categorical"])
print("Split      :", SPLIT["method"], f"blocks of {SPLIT['block_size_m']:.0f} m,",
      f"{SPLIT['n_folds']} folds, {SPLIT['test_fraction']:.0%} held out")
print("CA classes :", CLASSES, f"at {CA_SCALE} m")
print("  grouped  :", GROUPED_CLASSES, "->",
      {k: v for k, v in params["prediction"]["ca_markov"]["grouped_labels"].items()})
print("LULC years :", LULC_YEARS)
for _early, _late, _target in INTERVALS:
    print(f"  validate  : calibrate {_early}->{_late} ({_late - _early} yr), "
          f"project {(_target - _late) // (_late - _early)} step(s) to {_target}")
print("Scenarios  :", SCENARIOS)
print("Water      :", "masked from training" if RF["mask_water"] else "NOT masked",
      "| immutable classes excluded from scoring:",
      CA["validation"]["exclude_immutable"])
print()
print("Projection horizons:")
display(HORIZONS)
if (HORIZONS["offset_years"] != 0).any():
    print("*** At least one requested year is NOT a whole number of Markov steps.")
    print("*** Quote the EFFECTIVE year on every figure and in the report. A")
    print("*** fractional power of a transition matrix is not guaranteed to be a")
    print("*** transition matrix, so it is not computed.")

## Step 1 - probe the classifier before building anything on it

Notebook 04 settled the reducer output band names empirically before writing a single
export, and it saved a run. The same applies here, harder: `ee.Image.classify()` names
its output band `"classification"` by default even in REGRESSION mode, and
`ee.Classifier.explain()` returns a dictionary whose keys are not documented per
classifier. Guessing either would produce an export whose band means something other
than what the filename says.

This cell trains a tiny forest on a handful of pixels and prints exactly what came back.
**Read the output before moving on.** `prediction.project_lst_image` passes the band name
explicitly, so nothing here depends on the default - but if `explain()` does not carry an
`importance` key, Step 7's cross-check has nothing to compare against and should say so
rather than fail.

In [ ]:
# COLAB: RUN THIS CELL
_probe_region = aoi.cmc_boundary(params)
_probe_stack = prediction.prediction_stack(params, region=_probe_region)

print("prediction_stack bands:")
print(" ", _try_ee("band names", lambda: _probe_stack.bandNames().getInfo()))
print()

_probe_sample = prediction.training_sample_collection(
    params, region=_probe_region, n_pixels=300
)
_probe_n = _try_ee("probe sample size", lambda: _probe_sample.size().getInfo())
print("probe sample rows:", _probe_n)

if _probe_n:
    _probe_rf = prediction.fit_ee_rf(_probe_sample, params)
    _explained = _try_ee("explain", lambda: prediction.ee_rf_explain(_probe_rf), {})
    print()
    print("explain() keys:", sorted(_explained))
    if "importance" in _explained:
        print("importance predictors:", sorted(_explained["importance"]))
    else:
        print("*** explain() carries NO 'importance' key on this Earth Engine")
        print("*** version. Step 7's EE-vs-sklearn importance cross-check cannot")
        print("*** run; report that, and read the permutation importance alone.")
    print("outOfBagErrorEstimate:", _explained.get("outOfBagErrorEstimate"))

    _projected = prediction.project_lst_image(_probe_stack, _probe_rf, params)
    print()
    print("projected band name:", _try_ee("classified band",
                                          lambda: _projected.bandNames().getInfo()))
    print("  (named explicitly, NOT left as Earth Engine's default 'classification',")
    print("   which reads as a class code and is not what a regression output is)")
    _stats = _try_ee(
        "projected range",
        lambda: _projected.reduceRegion(
            reducer=ee.Reducer.minMax(), geometry=_probe_region,
            scale=MODEL_SCALE, maxPixels=params["composites"]["reduce_max_pixels"],
            tileScale=params["composites"]["tile_scale"],
        ).getInfo(),
    )
    print("projected LST range over the CMC (degC):", _stats)
    print()
    print("PASS: the server-side regression forest trains, explains and classifies.")
else:
    print("*** The probe sample is EMPTY. Nothing downstream can work.")
    print("*** Check that the epoch composite for", SOURCE, EPOCH, "has pixels.")

## Step 2 - probe Dynamic World coverage before choosing any year

Phase 5 measured Dynamic World "green" growing 5.4x from 2016 to 2024 over this district.
Almost none of it was land-cover change; it was Sentinel-2 coverage. A CA-Markov chain
calibrated across a coverage step would read the arrival of satellite data as the arrival
of trees.

Phase 5's probe covered 2016-2020 and 2024. **2021 has never been probed**, and 2021 is
the middle of the calibration triplet. So probe the whole range and read the answer before
calibrating anything.

The rule: every year in `LULC_YEARS` must clear
`prediction.ca_markov.min_observed_fraction`. If one does not, move the triplet, record
the new years in `PROGRESS.md` as the answer to S2, and re-run from Step 0.

In [ ]:
# COLAB: RUN THIS CELL
COVERAGE = _try_ee(
    "dynamic world coverage",
    lambda: spatial_stats.dynamic_world_coverage(
        params, WORK_REGION, years=[int(y) for y in CA["coverage_probe_years"]]
    ),
)
if COVERAGE is None:
    print("*** The coverage probe did not run. Do NOT calibrate on unprobed years.")
else:
    COVERAGE.to_csv("data/outputs/dynamic_world_coverage_2016_2025.csv", index=False)
    display(COVERAGE)

    _floor = float(CA["min_observed_fraction"])
    _needed = COVERAGE[COVERAGE["year"].isin(LULC_YEARS)]
    _short = _needed[_needed["observed_fraction"] < _floor]
    print()
    print(f"Years this phase depends on: {LULC_YEARS}")
    if _short.empty:
        print(f"PASS: every one clears the {_floor:.0%} observed-fraction floor.")
    else:
        print(f"*** {len(_short)} YEAR(S) BELOW THE {_floor:.0%} FLOOR:")
        for _row in _short.itertuples():
            print(f"      {_row.year}: {_row.observed_fraction:.1%}")
        print("*** Do NOT calibrate on these. Move prediction.ca_markov.")
        print("*** calibration_years / validation_year onto better-covered years,")
        print("*** record the change in PROGRESS.md as the answer to S2, and")
        print("*** re-run from Step 0.")

## Step 3 - submit the training sample

20 000 pixels at 100 m, carrying `x` and `y` in **EPSG:32644 metres**.

The coordinates are not decoration. Without them there is no spatially blocked split,
only a random one - and a random split of a raster sample puts pixels 100 m apart into
both train and test, so the reported R2 measures interpolation between neighbours rather
than prediction. `prediction.resolve_split` refuses `"random"` outright; Step 7 fits one
anyway, exactly once, to measure how much it would have flattered us.

This is a batch export, not a `.getInfo()`. Phase 5 established that a district-wide
reduction over a composite graph is not affordable interactively.

In [ ]:
# COLAB: RUN THIS CELL
TASKS = []

_task = prediction.export_training_sample(params, region=WORK_REGION)
TASKS.append(_task)
print("submitted:", _task.status().get("description"))
print("selectors:", prediction.training_sample_selectors(params))
print()
print(f"{len(TASKS)} task(s) queued so far.")

## Step 4 - submit the predictor raster

The same stack as the training sample, but as a **raster** on the 100 m grid.

This is what makes Part 2 runnable with no Earth Engine at all: with the observed
predictors on disk as arrays, the projected surface can be painted, scored and plotted in
numpy using the same scikit-learn model the blocked cross-validation validated. It also
means the projected map and the observed map are on one identical grid, so their
difference is a difference and not a resampling artefact.

Band order is passed explicitly. A GeoTIFF's band order is not self-describing, and a
silently reordered stack would rename every predictor and produce a plausible, wrong map.

In [ ]:
# COLAB: RUN THIS CELL
_task = prediction.export_predictor_raster(params, region=WORK_REGION)
TASKS.append(_task)
print("submitted:", _task.status().get("description"))
print("band order:", prediction.predictor_band_order(params))
print()
print(f"{len(TASKS)} task(s) queued so far.")

## Step 5 - submit the land-cover rasters

One per year in the calibration/validation/projection set, each with **two bands**.

`label` is the Dynamic World modal class, aggregated from 10 m to 100 m by mode.
`observed` is 1 where Dynamic World actually classified the cell and 0 where it never did.

The second band is not optional. A single-band 0-to-8 GeoTIFF cannot distinguish "class 0,
water" from "never classified", because masked pixels are written as 0 - and Phase 5 lost
a run to exactly that. Part 2 masks on `observed` before it computes a single transition.

These are also MOLUSCE's inputs; see `docs/molusce_handoff.md`.

In [ ]:
# COLAB: RUN THIS CELL
for _year in LULC_YEARS:
    _task = prediction.export_lulc_raster(params, _year, WORK_REGION)
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

print()
print(f"{len(TASKS)} task(s) queued so far.")

## Step 6 - submit the GHSL built-surface cross-check

`JRC/GHSL/P2023A/GHS_BUILT_S` runs to 2030, so the cellular automaton's projected built
area has an **independent** opinion for that horizon - built from a different sensor
lineage, by a different group, with a different method.

It is reported as agreement or disagreement. It is **not** used to correct the CA. Tuning
the automaton until it matches GHSL would turn an independent check into a fitted
parameter, and there would then be nothing left to check it with.

In [ ]:
# COLAB: RUN THIS CELL
_ghsl_year = int(params["prediction"]["ghsl_cross_check_year"])
_task = exports.image_to_drive(
    prediction.ghsl_built_cross_check(params).rename(["built_fraction"]).toFloat(),
    product="ghsl_built",
    aoi="district",
    params=params,
    region=WORK_REGION,
    band_order=["built_fraction"],
    scale_m=CA_SCALE,
    suffix=str(_ghsl_year),
)
TASKS.append(_task)
print("submitted:", _task.status().get("description"))
print()
print(f"{len(TASKS)} task(s) queued to Drive folder "
      f"'{params['exports']['drive_folder']}'.")
print("Re-run the NEXT cell until every state reads COMPLETED.")

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
_status = exports.describe_tasks(TASKS)
display(_status)

_state = _status["state"].value_counts().to_dict()
print("states:", _state)
_failed = _status[_status["state"] == "FAILED"]
if not _failed.empty:
    print()
    print(f"*** {len(_failed)} TASK(S) FAILED - these will never appear in Drive:")
    for _row in _failed.itertuples():
        print(f"  {_row.description}")
        print(f"    {_row.error}")
    print("*** Report the messages above; a failed export is a bug to fix, not")
    print("*** something to wait longer for.")

_done = int(_status["state"].eq("COMPLETED").sum())
print()
print(f"{_done} of {len(_status)} task(s) COMPLETED.")
if _done < len(_status) and _failed.empty:
    print("Still running. A district raster export is not a 30-second wait.")
    print("Re-run this cell rather than moving on.")
elif _done == len(_status):
    print("All done - copy them into data/interim/ per the WAIT HERE cell below.")

---
# WAIT HERE

**Part 1 is done once every task in the status cell reads `COMPLETED`** - that is one
training-sample CSV, one predictor GeoTIFF, one land-cover GeoTIFF per year, and the GHSL
cross-check raster.

Then get the files into `data/interim/`. Either:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/colombo_uhi_exports/*.csv data/interim/
!cp /content/drive/MyDrive/colombo_uhi_exports/*.tif data/interim/
```

or download them from Drive in a browser and upload into `data/interim/`.

**To run Part 2 as a separate session**, re-run only these, then continue below:

1. the **clone** cell,
2. the **pip** cell (skip if the runtime is fresh from Part 1),
3. the **params + auth** cell,
4. **Step 0** - it carries every import and constant Part 2 needs.

Part 2 touches Earth Engine only through Step 0's region guard, and nothing in it submits
a task.

---
# PART 2 - fit, validate and project (pure Python)

In [ ]:
# COLAB: RUN THIS CELL
# Locate the downloaded products and fail with an actionable message if absent.
# Filenames are RECONSTRUCTED with exports.export_name, never hardcoded, so Part 1
# and Part 2 cannot drift apart and look for a file the other never wrote.
SAMPLE_CSV = "data/interim/" + exports.export_name(
    "rf_training_sample", "district", params, res_m=MODEL_SCALE,
    suffix=f"{EPOCH}_{SOURCE}",
) + ".csv"
PREDICTOR_TIF = "data/interim/" + exports.export_name(
    "predictor_stack", "district", params, res_m=MODEL_SCALE,
    suffix=f"{EPOCH}_{SOURCE}",
) + ".tif"
LULC_TIF = {
    _year: "data/interim/" + exports.export_name(
        "lulc", "district", params, res_m=CA_SCALE, suffix=str(_year)
    ) + ".tif"
    for _year in LULC_YEARS
}
GHSL_TIF = "data/interim/" + exports.export_name(
    "ghsl_built", "district", params, res_m=CA_SCALE,
    suffix=str(params["prediction"]["ghsl_cross_check_year"]),
) + ".tif"

print("Found in data/interim/:")
for _f in sorted(glob.glob("data/interim/*")):
    print("  ", _f, f"({os.path.getsize(_f) / 1024:.0f} KB)")
print()

_wanted = {"training sample": SAMPLE_CSV, "predictor raster": PREDICTOR_TIF,
           "GHSL cross-check": GHSL_TIF}
_wanted.update({f"land cover {y}": p for y, p in LULC_TIF.items()})
_missing = [label for label, path in _wanted.items() if not os.path.exists(path)]
for _label, _path in _wanted.items():
    print(f"  {_label:22s}: {'OK     ' if os.path.exists(_path) else 'MISSING'} {_path}")

_essential = [
    label for label in _missing
    if label != "GHSL cross-check"  # the only optional one: it is a cross-check
]
if _essential:
    raise FileNotFoundError(
        f"missing {_essential}, and Part 2 cannot run without them.\n"
        "1. Check every Part 1 task reads COMPLETED.\n"
        f"2. Copy the files from Drive folder "
        f"'{params['exports']['drive_folder']}' into data/interim/ - see the\n"
        "   WAIT HERE cell above for the exact commands.\n"
        "3. If a filename looks close but not equal, Part 1 and this cell "
        "disagree\n   about a params value; re-run BOTH from the clone cell."
    )
print()
print("PASS: every essential Part 2 input is on disk.")

## Step 7 - Track A: fit the forest on spatially blocked data

The whole methodological weight of Track A sits in one line: the split cuts on **whole
2 km blocks**, never on rows.

Two pixels 100 m apart in Colombo are very nearly the same observation. A random split
puts one in train and one in test, so the model is scored on rows whose neighbours it has
memorised. The R2 that comes out is a measurement of interpolation, and it is always
higher than the number that matters.

Read the **fold-to-fold spread**, not just the mean. A large spread says the model's
performance depends on which part of the district it was asked about - which for a city
with a port, a wetland belt and a dense core is not a surprise, but is something the write-up
has to say.

In [ ]:
# COLAB: RUN THIS CELL
SAMPLE = prediction.read_training_sample(SAMPLE_CSV, params)
print(f"{len(SAMPLE):,} usable sampled pixels")
display(SAMPLE.head())

BLOCK_REPORT = prediction.require_enough_blocks(SAMPLE["block_id"], params)
print()
print("Spatial blocking:", BLOCK_REPORT)

CV = prediction.blocked_cv_scores(SAMPLE, params)
CV.to_csv("data/outputs/rf_blocked_cv_2020s.csv", index=False)
print()
print(f"Blocked {SPLIT['n_folds']}-fold cross-validation:")
display(CV)
print(f"  RMSE  {CV['rmse'].mean():.3f} degC  (fold range "
      f"{CV['rmse'].min():.3f} - {CV['rmse'].max():.3f})")
print(f"  R2    {CV['r2'].mean():.3f}        (fold range "
      f"{CV['r2'].min():.3f} - {CV['r2'].max():.3f})")
if CV["r2"].max() - CV["r2"].min() > 0.15:
    print("  *** The fold spread is wide. The model's skill depends on WHERE it")
    print("  *** is asked. Report the range, not the mean alone.")

# The single train/test split the reported model and every figure come from.
TRAIN_MASK, TEST_MASK = prediction.blocked_split(
    SAMPLE["block_id"].to_numpy(), SPLIT["test_fraction"], SPLIT["seed"]
)
FITTED = prediction.fit_sklearn_rf(SAMPLE, params, train_mask=TRAIN_MASK)
HELD_OUT = prediction.score_rows(FITTED, SAMPLE, TEST_MASK)
print()
print(f"Held-out blocks: n={HELD_OUT['n']:,}  RMSE={HELD_OUT['rmse']:.3f} degC  "
      f"R2={HELD_OUT['r2']:.3f}")

FIT_REPORT = prediction.build_validation_report(
    "lst_fit",
    {"rmse": HELD_OUT["rmse"], "r2": HELD_OUT["r2"]},
    params,
    held_out=True,
    n_train=FITTED["n_train"],
    n_test=HELD_OUT["n"],
    n_blocks=int(np.unique(SAMPLE["block_id"].to_numpy()[TEST_MASK]).size),
    block_size_m=SPLIT["block_size_m"],
    notes=[
        f"Fitted on {EPOCH} {SOURCE} at {MODEL_SCALE} m; a single-sensor "
        "(L8+L9) series, because the Collection 2 inter-calibration was tested "
        "over Colombo and failed.",
    ],
)
prediction.require_validated(FIT_REPORT, params)
print("PASS: the fitted surface has held-out validation metrics.")

_predicted = FITTED["model"].predict(SAMPLE.loc[TEST_MASK, PREDICTORS].to_numpy(float))
_fig = viz.plot_observed_vs_predicted(
    SAMPLE.loc[TEST_MASK, RESPONSE].to_numpy(float), _predicted,
    "figures/rf_observed_vs_predicted_2020s.png", params, FIT_REPORT,
    title=f"Random forest LST, {EPOCH} {SOURCE}, held-out spatial blocks",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

## Step 8 - measure what a random split would have flattered us by

This is the only cell in the project that fits a random split, and it exists to produce a
number for the write-up rather than a result.

Both rows below see the same pixels and the same model. The only difference is where the
line between train and test was drawn. Whatever gap appears is the amount by which a
conventionally-reported R2 would have overstated this model - and it is why
`prediction.split.method` may not be set to `"random"` at all.

The permutation importance below it is computed on the **held-out** blocks. Impurity
importance (`model.feature_importances_`) is biased toward predictors with many distinct
values, and `lcz_class` is exactly that, so it would be flattered for a reason that has
nothing to do with what it contributes.

In [ ]:
# COLAB: RUN THIS CELL
SPLIT_COMPARISON = prediction.compare_split_strategies(SAMPLE, params, n_repeats=5)
SPLIT_COMPARISON.to_csv("data/outputs/rf_split_comparison_2020s.csv", index=False)
display(SPLIT_COMPARISON.round(4))

_by = SPLIT_COMPARISON.set_index("method")
_gap = float(_by.loc["random", "r2_mean"] - _by.loc["spatial_block", "r2_mean"])
_noise = float(
    (_by.loc["random", "r2_sd"] ** 2 + _by.loc["spatial_block", "r2_sd"] ** 2) ** 0.5
)
print()
print(f"Random   R2 {_by.loc['random', 'r2_mean']:.3f} "
      f"+/- {_by.loc['random', 'r2_sd']:.3f} over "
      f"{int(_by.loc['random', 'n_repeats'])} splits")
print(f"Blocked  R2 {_by.loc['spatial_block', 'r2_mean']:.3f} "
      f"+/- {_by.loc['spatial_block', 'r2_sd']:.3f}")
print(f"Gap: {_gap:+.3f} R2, against a combined spread of {_noise:.3f}.")
print()
if abs(_gap) < _noise:
    print("*** THE GAP IS SMALLER THAN THE SPREAD IT IS DRAWN FROM. This")
    print("*** comparison cannot resolve the leakage effect at this sample")
    print("*** size and block size. Report it as inconclusive - NOT as evidence")
    print("*** that leakage is absent - and run a larger")
    print("*** prediction.split.block_size_m as a sensitivity. Run 2 got")
    print("*** -0.037 from a single pair of splits while the blocked CV's own")
    print("*** folds ranged 0.761 to 0.887; that number meant nothing.")
elif _gap <= 0:
    print("*** The random split scored LOWER even across repeats. With 2 km")
    print("*** blocks in a 700 km2 district adjacent blocks are still")
    print("*** neighbours, so the blocked split is a partial control. Say so,")
    print("*** and run a larger prediction.split.block_size_m.")
else:
    print(f"Spatial leakage inflates R2 by {_gap:+.3f}, beyond the spread.")
    print("This is the number the write-up should quote for why every reported")
    print("metric in this phase comes from blocked held-out data.")
print()
print("ONLY the spatial_block row is reportable as model performance.")

IMPORTANCE = prediction.permutation_importance_frame(
    FITTED, SAMPLE, params, mask=TEST_MASK
)
IMPORTANCE.to_csv("data/outputs/rf_permutation_importance_2020s.csv", index=False)
print()
display(IMPORTANCE)

_fig = viz.plot_feature_importance(
    IMPORTANCE, "figures/rf_permutation_importance_2020s.png", params, FIT_REPORT,
    title=f"Permutation importance on held-out blocks ({EPOCH}, {MODEL_SCALE} m)",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

print()
print("NOTE: lcz_class enters as an INTEGER code "
      f"(prediction.rf.lcz_encoding = {RF['lcz_encoding']!r}), so the Earth Engine")
print("and scikit-learn fits stay comparable. Tree ensembles can partition an")
print("integer-coded categorical, but the coding is arbitrary - do not read the")
print("importance of lcz_class as the importance of any particular zone type.")

## Step 9 - Track B: calibrate the chain and validate it on a held-out year

Calibrate on **2018 -> 2021**, project one step to **2024**, and score against the
*observed* 2024 map. Calibrating and validating on the same pair would measure
memorisation.

Four numbers come out, and they must be read together:

* **Kappa** - CLAUDE.md requires it, and on its own it is misleading here.
* **Kappa of the no-change null** - what a projection that simply copies the 2021 map
  scores. Over three years most cells persist, so this is normally high. Kappa above it is
  the only part that is skill.
* **Pontius quantity / allocation split** - whether the error is getting the *amount* of
  each class wrong, or the amount right and the *place* wrong.
* **Figure of merit** - scored on the cells that actually changed. A no-change projection
  scores exactly 0 here, which is the point.

Everything is masked to cells Dynamic World observed in **all three** years. A cell that
appeared in 2024 because Sentinel-2 coverage improved is not a land-cover transition.

### Two things run 1 changed here

**Immutable classes are excluded from scoring.** Water is configured immutable, so the CA
is *definitionally* correct on every water cell - including them does not measure
agreement, it pads the diagonal. Run 1 scored 32 % water and read Kappa 0.928 against a
no-change null of 0.933.

**Both class schemes are validated.** Run 1's transition matrix showed Dynamic World
flipping between spectrally similar vegetation classes: grass persisted only **0.399**
with **0.425** going to trees, shrub persisted **0.364** with **0.484** going to trees. A
42 % grass-to-trees transition in three years is the classifier changing its mind, not the
land cover changing - and because a Markov-CA allocates **net** demand, it structurally
cannot reproduce that gross churn (run 1: 9,479 projected changes against 67,831
observed). So the run below scores the raw 7-class scheme *and* a grouped scheme that
merges trees, grass and shrub into one "green" class, and reports the pair.

**Two intervals, not one.** Run 2 showed the automaton reproducing the *quantity* of
change well (disagreement 0.011) while failing to *place* it (allocation 0.069, figure of
merit 0.03) — and grouping barely moved it, 0.028 to 0.032. That is the signature of a
**net-demand allocator facing gross churn**, not of classifier noise alone: the CA
projected about 936 changes against 8,832 observed. A longer step lets real change
accumulate against a roughly fixed level of classifier noise, so the second interval
(2017 → 2021, validated at 2025) is a direct test of which explanation holds.

**The projection always uses the UNGROUPED scheme.** Run 2 auto-selected by figure of
merit and picked `grouped` on 0.032 against 0.028 — choosing between two failures — and
that choice silently reduced the greening scenario to converting **zero** cells, because
grouping leaves only bare land eligible and the district has 67 bare cells in total.
Grouping is a *validation sensitivity*: it answers "is the failure caused by classifier
churn?" and nothing else.

In [ ]:
# COLAB: RUN THIS CELL
LULC = {}
LULC_OBSERVED = {}
for _year, _path in LULC_TIF.items():
    _labels, _observed, LULC_PROFILE = prediction.read_lulc_raster(_path, params)
    LULC[_year] = _labels
    LULC_OBSERVED[_year] = _observed
    print(f"{_year}: {_labels.shape} cells, "
          f"{_observed.mean():.1%} classified by Dynamic World")

# Only cells Dynamic World saw in EVERY year take part. A cell that "appeared" in
# 2024 because coverage improved is not a transition.
VALID = np.logical_and.reduce([LULC_OBSERVED[y] for y in LULC_YEARS])
VALID &= np.logical_and.reduce([np.isin(LULC[y], CLASSES) for y in LULC_YEARS])
print()
print(f"Cells usable in all {len(LULC_YEARS)} years: {VALID.sum():,} of {VALID.size:,} "
      f"({VALID.mean():.1%})")
print(f"One cell is {CA_SCALE} m x {CA_SCALE} m = {CA_SCALE ** 2 / 1e4:.2f} ha.")

# Step 0's guard measured the REGION and passed; run 2 still analysed 1,256 km2,
# because Export.image.toDrive writes the region's BOUNDING BOX and nothing
# clipped to the geometry. 46 % of that raster lay outside Colombo District.
# This is the raster-side companion to the Step 0 check, and it is the one that
# would have caught it.
AREA_CHECK = prediction.require_expected_area(int(VALID.sum()), params)
print(f"PASS: analysis grid {AREA_CHECK['area_km2']:,.1f} km2, "
      f"{AREA_CHECK['departure']:+.1%} from the expected "
      f"{AREA_CHECK['expected_km2']:,.0f} km2 for Colombo District.")

_early, _late = CALIBRATION
_steps = (VALIDATION_YEAR - _late) // (_late - _early)
print()
print(f"Calibrating {_early} -> {_late} ({_late - _early} yr), projecting "
      f"{_steps} step(s) to {VALIDATION_YEAR}.")

CALIBRATED = prediction.ca_markov_project(
    LULC[_early], LULC[_late], params, steps=_steps, changeable=VALID
)
TRANSITIONS = pd.DataFrame(
    CALIBRATED["probabilities"], index=CLASSES, columns=CLASSES
)
TRANSITIONS.to_csv(f"data/outputs/transition_probabilities_{_early}_{_late}.csv")
print()
print(f"One-step transition probabilities, {_early} -> {_late}:")
display(TRANSITIONS.round(3))
print("Persistence (the diagonal):",
      {c: round(float(TRANSITIONS.loc[c, c]), 3) for c in CLASSES})

_fig = viz.plot_transition_matrix(
    CALIBRATED["probabilities"],
    f"figures/transition_matrix_{_early}_{_late}.png", params, CLASSES,
    title=f"Dynamic World transitions, {_early} to {_late} ({CA_SCALE} m)",
)
print()
print("Read the DIAGONAL. Run 1 found grass persisting at only 0.399 and shrub")
print("at 0.364, with 0.425 and 0.484 respectively going to trees. A 42 %")
print("grass-to-trees transition in three years is the classifier changing its")
print("mind, not the land cover changing - which is why the next cell scores a")
print("grouped scheme alongside the raw one.")
print("Wrote", _fig)
display(Image(filename=str(_fig)))

In [ ]:
# COLAB: RUN THIS CELL
# Score BOTH class schemes against the OBSERVED validation year, on the cells the
# CA could actually have changed.
SCHEMES = {"ungrouped": CLASSES}
if CA.get("report_grouped_sensitivity", True):
    SCHEMES["grouped"] = GROUPED_CLASSES


def _recoder(scheme):
    # Bound at call time, NOT captured from the loop variable - a late-binding
    # closure here would silently apply the last scheme to every row.
    if scheme == "ungrouped":
        return lambda a: a
    return lambda a: prediction.group_classes(a, params)


VALIDATION = {}
_rows = []
for _early, _late, _target in INTERVALS:
    _step = _late - _early
    _n_steps = (_target - _late) // _step
    _run = prediction.ca_markov_project(
        LULC[_early], LULC[_late], params, steps=_n_steps, changeable=VALID
    )
    for _scheme, _codes in SCHEMES.items():
        _recode = _recoder(_scheme)
        _initial = _recode(LULC[_late])
        _observed = _recode(LULC[_target])
        _projected = _recode(_run["labels"])

        # Immutable classes are excluded: the CA is forbidden from changing
        # them, so scoring them measures the constraint, not the model.
        _score_mask, _mask_report = prediction.scoring_mask(
            params, _initial, _observed
        )
        _score_mask &= VALID

        _metrics = prediction.validate_projection(
            _initial, _observed, _projected, params, _codes, mask=_score_mask
        )
        VALIDATION[(_early, _late, _target, _scheme)] = {
            "metrics": _metrics, "mask": _score_mask,
            "mask_report": _mask_report, "classes": _codes, "recode": _recode,
            "run": _run,
        }
        _rows.append({
            "calibrate": f"{_early}-{_late}", "step_years": _step,
            "validate": _target, "scheme": _scheme, "n_classes": len(_codes),
            "n_excluded": _mask_report["n_excluded"], **_metrics,
        })

LULC_SENSITIVITY = pd.DataFrame(_rows)
LULC_SENSITIVITY.to_csv(
    f"data/outputs/lulc_validation_{VALIDATION_YEAR}.csv", index=False
)
display(LULC_SENSITIVITY[[
    "calibrate", "step_years", "validate", "scheme", "n_classes", "n_scored",
    "kappa", "persistence_kappa", "kappa_above_null", "figure_of_merit",
    "quantity_disagreement", "allocation_disagreement",
]].round(4))

_floor_k = float(CA["validation"]["min_kappa"])
_floor_f = float(CA["validation"]["min_figure_of_merit"])
for _key, _entry in VALIDATION.items():
    _early, _late, _target, _scheme = _key
    _m = _entry["metrics"]
    print()
    print(f"--- {_early}->{_late} -> {_target} | {_scheme} "
          f"({len(_entry['classes'])} classes) ---")
    print(f"  scored {_m['n_scored']:,} cells; "
          f"{_entry['mask_report']['n_excluded']:,} excluded as immutable "
          f"(classes {_entry['mask_report']['immutable_classes']})")
    print(f"  Kappa {_m['kappa']:.3f} against a no-change null of "
          f"{_m['persistence_kappa']:.3f} -> {_m['kappa_above_null']:+.3f} "
          f"beyond copying the {_late} map")
    print(f"  Figure of merit {_m['figure_of_merit']:.3f} on "
          f"{_m['observed_change']:,} changed cells ({_m['hits']:,} hits, "
          f"{_m['misses']:,} misses, {_m['false_alarms']:,} false alarms, "
          f"{_m['wrong_hits']:,} wrong class)")
    print(f"  Error split: {_m['quantity_disagreement']:.3f} quantity vs "
          f"{_m['allocation_disagreement']:.3f} allocation")
    if _m["kappa"] >= _floor_k and _m["figure_of_merit"] >= _floor_f:
        print(f"  PASS: Kappa >= {_floor_k} and figure of merit >= {_floor_f}.")
    else:
        print(f"  *** BELOW A FLOOR (min_kappa={_floor_k}, "
              f"min_figure_of_merit={_floor_f}).")
    if _m["kappa_above_null"] <= 0:
        print("  *** THE PROJECTION IS NO BETTER THAN DOING NOTHING. A")
        print("  *** no-change map scores at least as well. Whatever else this")
        print("  *** scheme is used for, it has demonstrated no skill at")
        print("  *** locating change.")

print()
print("*** These are FLOORS, NOT TARGETS. Report a failure and what it means for")
print("*** the projection - do NOT tune the automaton until it clears them. A CA")
print("*** fitted to its own validation set has no validation set.")
print()
print("Does a LONGER STEP help? Compare the two calibrate rows at the same")
print("scheme. If the figure of merit barely moves, the failure is structural -")
print("a net-demand allocator cannot reproduce gross churn - rather than")
print("classifier noise that a longer interval would average out.")
print()
print("Record in PROGRESS.md as S9: what the two intervals and two schemes say.")

# The projection ALWAYS uses the ungrouped scheme on the primary interval.
# Run 2 auto-selected by figure of merit, picked `grouped` on 0.032 against
# 0.028 - choosing between two failures - and that silently reduced the greening
# scenario to converting ZERO cells, because grouping leaves only bare land
# eligible and the district holds 67 bare cells in total. Grouping is a
# validation SENSITIVITY; it answers "is this classifier churn?" and nothing
# more.
PROJECTION_SCHEME = "ungrouped"
GROUPED = False
PRIMARY = (*INTERVALS[0], PROJECTION_SCHEME)
PROJECTION_CLASSES = VALIDATION[PRIMARY]["classes"]
LULC_METRICS = VALIDATION[PRIMARY]["metrics"]
CALIBRATED = VALIDATION[PRIMARY]["run"]
print()
print(f"Projecting with the {PROJECTION_SCHEME.upper()} scheme on the primary "
      f"interval {PRIMARY[0]}-{PRIMARY[1]} -> {PRIMARY[2]}.")
print("Grouping stays a sensitivity; it does not decide the projection.")

LULC_REPORT = prediction.build_validation_report(
    "lulc_projection", LULC_METRICS, params,
    held_out=True, n_test=int(LULC_METRICS["n_scored"]),
    notes=[
        f"Calibrated {PRIMARY[0]}-{PRIMARY[1]}, projected to {PRIMARY[2]} and "
        f"scored against the observed map using the {PROJECTION_SCHEME} class "
        f"scheme on the {LULC_METRICS['n_scored']:,} cells Dynamic World "
        "classified in every year AND that the CA was permitted to change.",
        f"Kappa of the no-change null is "
        f"{LULC_METRICS['persistence_kappa']:.3f}; read the model's Kappa "
        "against that, not against zero.",
    ],
)
prediction.require_validated(LULC_REPORT, params)
print("PASS: the land-cover projection has held-out validation metrics.")

_entry = VALIDATION[PRIMARY]
_late, _target = PRIMARY[1], PRIMARY[2]
_show = _entry["mask"]
_fig = viz.plot_lulc_validation(
    np.where(_show, _entry["recode"](LULC[_late]), -1),
    np.where(_show, _entry["recode"](LULC[_target]), -1),
    np.where(_show, _entry["recode"](CALIBRATED["labels"]), -1),
    f"figures/lulc_validation_{_target}.png", params, LULC_REPORT,
    title=(f"CA-Markov {_late} -> {_target}, {PROJECTION_SCHEME} scheme "
           f"({CA_SCALE} m)"),
    class_labels=(
        {int(k): str(v) for k, v in CA["grouped_labels"].items()}
        if GROUPED else None
    ),
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

## Step 10 - recalibrate on the full span and project the horizons

Validation is done; now recalibrate on **2018 -> 2024** and project forward. One Markov
step is the calibrated interval itself, so a step here is 6 years.

`prediction.resolve_projection_steps` printed the horizon table back in Step 0. A
requested year that is not a whole number of steps lands on a **different, effective**
year, and everything from here on quotes the effective year. A fractional power of a
transition matrix is not guaranteed to be a valid transition matrix, so it is not
computed - and a 1.83-step projection is not quietly labelled with the year that was
asked for.

In [ ]:
# COLAB: RUN THIS CELL
_base_early, _base_late = BASE_YEARS
print(f"Recalibrating on {_base_early} -> {_base_late} "
      f"({_base_late - _base_early} yr per Markov step).")

_recode = VALIDATION[PROJECTION_SCHEME]["recode"]
print(f"Projecting with the {PROJECTION_SCHEME} class scheme: "
      f"{PROJECTION_CLASSES}")

PROJECTED = {}
AREAS = {}
for _row in HORIZONS.itertuples():
    _result = prediction.ca_markov_project(
        _recode(LULC[_base_early]), _recode(LULC[_base_late]), params,
        steps=int(_row.steps), changeable=VALID, grouped=GROUPED,
    )
    PROJECTED[int(_row.effective_year)] = _result
    AREAS[int(_row.effective_year)] = _result["areas"]
    _label = (f"{_row.effective_year}" if _row.offset_years == 0
              else f"{_row.effective_year} (asked for {_row.requested_year})")
    print()
    print(f"--- {_label}: {_row.steps} step(s) ---")
    display(_result["areas"])
    _short = int(_result["allocation"]["shortfall"].sum())
    if _short:
        print(f"  *** {_short:,} cell(s) of demand could not be allocated - there")
        print("  *** were not enough changeable donor cells. Reported, not hidden.")

_all_areas = pd.concat(
    [frame.assign(year=year) for year, frame in AREAS.items()], ignore_index=True
)
_all_areas.to_csv("data/outputs/projected_class_areas.csv", index=False)
print()
print("Wrote data/outputs/projected_class_areas.csv")

# Independent opinion on the built area, if the GHSL raster came through.
_ghsl_year = int(params["prediction"]["ghsl_cross_check_year"])
if os.path.exists(GHSL_TIF) and _ghsl_year in PROJECTED:
    import rasterio

    with rasterio.open(GHSL_TIF) as _handle:
        _ghsl = _handle.read(1, masked=True).filled(np.nan)

    GHSL_COMPARISON = prediction.ghsl_built_comparison(
        _ghsl, PROJECTED[_ghsl_year]["labels"], params,
        built_class=6, cell_area_m2=CA_SCALE ** 2, mask=VALID,
    )
    pd.DataFrame([GHSL_COMPARISON]).to_csv(
        f"data/outputs/ghsl_cross_check_{_ghsl_year}.csv", index=False
    )
    print()
    print(f"--- GHSL {_ghsl_year} cross-check ---")
    print("COMPARABLE PAIR - built-DOMINANT cells, GHSL thresholded at "
          f"{GHSL_COMPARISON['threshold']:.2f}:")
    print(f"  GHSL      : {GHSL_COMPARISON['ghsl_dominant_cells']:,} cells "
          f"({GHSL_COMPARISON['ghsl_dominant_km2']:.1f} km2)")
    print(f"  CA-Markov : {GHSL_COMPARISON['ca_built_cells']:,} cells "
          f"({GHSL_COMPARISON['ca_built_km2']:.1f} km2)")
    print(f"  Difference: {GHSL_COMPARISON['cell_difference']:+,} cells; the two "
          f"agree on {GHSL_COMPARISON['cell_agreement']:.1%} of cells")
    print()
    print("DIFFERENT QUANTITY, reported separately and NOT differenced against")
    print("the pair above:")
    print(f"  GHSL fraction-weighted built SURFACE: "
          f"{GHSL_COMPARISON['ghsl_fraction_weighted_km2']:.1f} km2")
    print("  Run 1 compared this against the CA's built-DOMINANT area and")
    print("  reported a 232 % disagreement that was an artefact of comparing a")
    print("  fraction to a modal class.")
    print()
    print("Reported as agreement or disagreement. NOT used to correct the CA -")
    print("tuning the automaton to match GHSL would turn an independent check")
    print("into a fitted parameter, leaving nothing to check it with.")
elif not os.path.exists(GHSL_TIF):
    print()
    print("GHSL cross-check raster absent; skipping. Not a failure - it is a")
    print("cross-check, and its absence costs a comparison, not a result.")
else:
    print()
    print(f"No projection lands on {_ghsl_year}, so there is nothing to compare")
    print("GHSL against. Check the horizon table in Step 0.")

## Step 11 - interim priority zones, and the greening scenario

Phase 7's MCDA/AHP weighted overlay does not exist yet, so priority zones come from a
**Phase-5 interim proxy**: a weighted rank of Getis-Ord hot-spot z, 2020s mean LST, and
inverted NDVI, over the 557 GN divisions.

Two honest limits on that rule, both of which belong in the write-up:

* it ranks on **outcome**, not **feasibility** - it says nothing about land ownership,
  existing use, or whether anything can actually be planted there. That is precisely what
  Phase 7 adds;
* it inherits `caveats.zonal_not_pixel`. A GN division is an administrative unit, and a
  ranking across 557 of them is a property of that aggregation.

`prediction.apply_greening_scenario` takes a plain zone list, so Phase 7 swaps its output
in without touching any scenario code. The greening lever itself is
`prediction.scenarios.greening.canopy_increase_fraction`, and **built land is protected**:
converting it to canopy is a demolition programme, not a greening programme, and modelling
it silently would overstate what is achievable.

In [ ]:
# COLAB: RUN THIS CELL
import geopandas as gpd
import rasterio.features

_zone_path = "data/outputs/gn_divisions_colombo.geojson"
if not os.path.exists(_zone_path):
    raise FileNotFoundError(
        f"{_zone_path} is missing. It is COMMITTED by Phase 5, so a fresh clone "
        "has it; if it is absent here, the clone cell did not run or the repo is "
        "at a pre-Phase-5 revision."
    )
# read_priority_geometry, NOT spatial_stats.read_zone_geodataframe. The Phase 5
# reader expects the RAW export keyed on adm4_pcode; the committed file is Phase
# 5's POST-PROCESSED output, in which that column is already zone_id. Colab run 1
# died here, at this exact line.
ZONES = prediction.read_priority_geometry(_zone_path, params)
print(f"{len(ZONES)} GN divisions")

# Rasterise zone INDEX (not pcode) onto the model grid, once. It serves both the
# per-zone criterion means below and the priority mask below that.
_index = {zid: i for i, zid in enumerate(ZONES["zone_id"])}
ZONE_INDEX = rasterio.features.rasterize(
    ((geom, _index[zid]) for geom, zid in zip(ZONES.geometry, ZONES["zone_id"])),
    out_shape=VALID.shape, transform=LULC_PROFILE["transform"],
    fill=-1, dtype="int32",
)
print(f"{(ZONE_INDEX >= 0).sum():,} of {ZONE_INDEX.size:,} cells fall inside a division")

# Per-zone means of the criteria, straight off the predictor raster - so they are
# on exactly the grid the projection runs on, with no join to go wrong.
PREDICTOR_ARRAYS, PREDICTOR_PROFILE = prediction.read_predictor_raster(
    PREDICTOR_TIF, params
)
if PREDICTOR_ARRAYS[RESPONSE].shape != VALID.shape:
    raise ValueError(
        f"the predictor raster is {PREDICTOR_ARRAYS[RESPONSE].shape} but the "
        f"land-cover rasters are {VALID.shape}. They must be one grid - "
        "re-export both from Part 1 with the same region and scale."
    )


def _zone_mean(array):
    flat = np.asarray(array, dtype=float).ravel()
    keys = ZONE_INDEX.ravel()
    keep = (keys >= 0) & np.isfinite(flat)
    total = np.bincount(keys[keep], weights=flat[keep], minlength=len(ZONES))
    count = np.bincount(keys[keep], minlength=len(ZONES))
    return np.where(count > 0, total / np.maximum(count, 1), np.nan)


CRITERIA = pd.DataFrame({
    "zone_id": ZONES["zone_id"].to_numpy(),
    RESPONSE: _zone_mean(PREDICTOR_ARRAYS[RESPONSE]),
    "NDVI": _zone_mean(PREDICTOR_ARRAYS["NDVI"]),
})

# Getis-Ord z comes from the committed Phase 5 table.
_gi_path = f"data/outputs/gi_star_gn_{params['prediction']['priority_zones']['epoch']}.csv"
_gi = pd.read_csv(_gi_path)
_gi["zone_id"] = _gi["zone_id"].astype(str)
_z_column = next(
    (c for c in ("gi_z", "z", "gi_star_z", "z_score") if c in _gi.columns), None
)
if _z_column is None:
    raise ValueError(
        f"{_gi_path} has no recognisable Gi* z column; it has "
        f"{sorted(_gi.columns)}. Name the right one here rather than guessing."
    )
print(f"Using {_z_column!r} from {_gi_path} as the hot-spot criterion.")
CRITERIA = CRITERIA.merge(
    _gi[["zone_id", _z_column]].rename(columns={_z_column: "gi_z"}),
    on="zone_id", how="left",
)
print(f"{CRITERIA['gi_z'].isna().sum()} division(s) with no Gi* value "
      "(ranked last on that criterion, not dropped)")

PRIORITY = prediction.interim_priority_zones(CRITERIA, params)
PRIORITY.to_csv("data/outputs/greening_priority_zones_interim.csv", index=False)
print()
print(f"Top {int(PRIORITY['priority'].sum())} of {len(PRIORITY)} divisions flagged:")
display(PRIORITY.head(10))

_priority_ids = set(PRIORITY.loc[PRIORITY["priority"], "zone_id"])
PRIORITY_MASK = np.isin(
    ZONE_INDEX, [_index[z] for z in _priority_ids if z in _index]
)
print(f"Priority mask covers {PRIORITY_MASK.sum():,} cells "
      f"({PRIORITY_MASK.sum() * CA_SCALE ** 2 / 1e6:.1f} km2)")

In [ ]:
# COLAB: RUN THIS CELL
# Apply each scenario to each projected horizon.
SCENARIO_LABELS = {}
SCENARIO_REPORTS = []
for _year, _result in PROJECTED.items():
    for _name in SCENARIOS:
        _labels, _report = prediction.apply_greening_scenario(
            _result["labels"], PRIORITY_MASK & VALID, params, _name,
            grouped=GROUPED,
        )
        SCENARIO_LABELS[(_name, _year)] = _labels
        SCENARIO_REPORTS.append(
            {"year": _year, "scheme": PROJECTION_SCHEME, **_report}
        )

SCENARIO_TABLE = pd.DataFrame(SCENARIO_REPORTS)
SCENARIO_TABLE.to_csv("data/outputs/scenario_conversions.csv", index=False)
display(SCENARIO_TABLE)

_greening = prediction.resolve_scenario("greening", params, grouped=GROUPED)
print()
print(f"The greening lever ({PROJECTION_SCHEME} scheme): convert "
      f"{_greening['canopy_increase_fraction']:.0%} of eligible cells")
print(f"(classes {_greening['eligible_classes']}) inside priority zones to class "
      f"{_greening['target_class']}.")
print(f"Protected: {_greening['protect_classes']} - water, and BUILT land, because")
print("converting built land to canopy is a demolition programme, not a greening")
print("programme.")
if GROUPED:
    print()
    print("*** UNDER GROUPING THE LEVER IS WEAKER. Grass and shrub already count")
    print("*** as 'green', so only bare land is eligible. That is a real")
    print("*** consequence of suppressing the classifier churn, not a modelling")
    print("*** choice to hide - state it whenever quoting a greening effect from")
    print("*** this run.")
    print(f"*** Converted cells are painted with the class "
          f"{_greening['paint_as_class']} (TREES) profile, not the blended")
    print("*** 'green' median. Assumption: new planting is canopy, not lawn.")
    print("*** Canopy and lawn differ thermally and the grouped class cannot")
    print("*** tell them apart.")
print()
print("Cells are chosen by highest neighbourhood share of existing canopy, not at")
print("random: planting that extends existing canopy is both more plausible as")
print("policy and more effective thermally than the same area scattered as")
print("isolated cells. That choice is a MODELLING ASSUMPTION, not a finding.")

## Step 12 - projected land cover to projected LST

Each projected class map is painted with the predictor values that class carries *today*,
and the result goes through the forest from Step 7.

**This is a substitution, not a simulation.** It assumes the spectral and structural
signature of "built" in 2030 is the signature of "built" in the training epoch.
Densification inside a class, or a change in construction materials, is invisible to it.

What is projected and what is not:

| Predictor | Treatment |
|---|---|
| NDVI, NDBI, built_fraction | painted from the projected land cover |
| elevation_m, dist_coast_km | genuinely static |
| lcz_class | **held fixed and flagged** - it is not strictly static, but no free layer updates it, so it is held rather than invented |
| pop_density | **held constant at WorldPop 2020**, which is where that dataset ends. A projected population would be a second unvalidated model stacked on the first |

Then the extrapolation check. A random forest returns the mean of the training rows in
each leaf, so a projected pixel whose drivers lie outside the training envelope is pinned
to the edge of what the model has seen - a systematic error in a known direction, not
noise. `prediction.require_validated` fails the product above
`prediction.rf.extrapolation.tolerance_fraction`.

In [ ]:
# COLAB: RUN THIS CELL
# What each observed class looks like today, on the same grid the projection runs
# on. The profile is built on the SCHEME THE PROJECTION USES, so a grouped run
# paints grouped classes rather than silently looking up codes that no longer
# exist in the projected map.
_observed_frame = prediction.raster_sample_frame(
    {name: PREDICTOR_ARRAYS[name] for name in PREDICTORS},
    labels=np.where(VALID, _recode(LULC[_base_late]), -1),
)
_observed_frame = _observed_frame[_observed_frame["lulc_class"] >= 0]

DYNAMIC = [n for n in ("NDVI", "NDBI", "built_fraction") if n in PREDICTORS]
STATIC = [n for n in PREDICTORS if n not in DYNAMIC]
CLASS_PROFILE = prediction.class_conditional_predictors(
    _observed_frame, "lulc_class", DYNAMIC
)

# Under grouping, "green" blends canopy and lawn, which differ thermally. The
# greening scenario therefore paints its converted cells with the profile of a
# NAMED class (paint_as_class), taken from the UNGROUPED observed map so it is
# genuinely canopy. Stated on the figure as an assumption, not a finding.
_ungrouped_frame = prediction.raster_sample_frame(
    {name: PREDICTOR_ARRAYS[name] for name in PREDICTORS},
    labels=np.where(VALID, LULC[_base_late], -1),
)
CANOPY_PROFILE = prediction.class_conditional_predictors(
    _ungrouped_frame[_ungrouped_frame["lulc_class"] >= 0], "lulc_class", DYNAMIC
)
CLASS_PROFILE.to_csv("data/outputs/class_conditional_predictors.csv", index=False)
print("Class-conditional predictor medians (the substitution table):")
display(CLASS_PROFILE)
if CLASS_PROFILE["thin"].any():
    print("*** Thinly-sampled class(es):",
          CLASS_PROFILE.loc[CLASS_PROFILE["thin"], "class_code"].tolist())
    print("*** They are still painted - a projected cell needs SOME value - but")
    print("*** their median rests on few pixels. Name them in the report.")
print()
print("Projected:", DYNAMIC, "| held fixed:", STATIC)
print("pop_density is WorldPop 2020, where that dataset ends "
      f"({params['prediction']['held_constant']}).")
print(f"Profile scheme: {PROJECTION_SCHEME}. Greening-converted cells are "
      "repainted with the UNGROUPED")
print("canopy profile, so 'new planting is canopy, not lawn' is an explicit "
      "assumption rather")
print("than an accident of which classes happened to be merged.")

SURFACES = {}
EXTRAPOLATION = {}
CONVERTED = {}
for (_name, _year), _labels in SCENARIO_LABELS.items():
    _painted = prediction.paint_class_predictors(_labels, CLASS_PROFILE, DYNAMIC)

    # Cells this scenario converted, repainted with the canopy profile.
    _scenario = prediction.resolve_scenario(_name, params, grouped=GROUPED)
    _changed = _labels != PROJECTED[_year]["labels"]
    CONVERTED[(_name, _year)] = int(_changed.sum())
    if _changed.any() and _scenario["paint_as_class"] is not None:
        _canopy = CANOPY_PROFILE.set_index("class_code")
        _code = int(_scenario["paint_as_class"])
        if _code in _canopy.index:
            for _band in DYNAMIC:
                _painted[_band] = np.where(
                    _changed, float(_canopy.loc[_code, _band]), _painted[_band]
                )

    _stack = {**{n: PREDICTOR_ARRAYS[n] for n in STATIC}, **_painted}
    _stack = {n: np.where(VALID, _stack[n], np.nan) for n in PREDICTORS}
    SURFACES[(_name, _year)] = prediction.predict_surface(FITTED, _stack, params)
    EXTRAPOLATION[(_name, _year)] = prediction.extrapolation_flags(
        SAMPLE.loc[TRAIN_MASK],
        prediction.raster_sample_frame(_stack).dropna(),
        params,
    )

EXTRAP_TABLE = pd.DataFrame([
    {"scenario": k[0], "year": k[1], "fraction_outside": v["fraction"],
     "n": v["n"], "within_tolerance": v["within_tolerance"],
     **{f"outside_{p}": s for p, s in v["by_predictor"].items()}}
    for k, v in EXTRAPOLATION.items()
])
EXTRAP_TABLE.to_csv("data/outputs/extrapolation_check.csv", index=False)
print()
print("Extrapolation beyond the training envelope:")
display(EXTRAP_TABLE)
if not EXTRAP_TABLE["within_tolerance"].all():
    print("*** At least one scenario exceeds "
          f"{params['prediction']['rf']['extrapolation']['tolerance_fraction']:.0%}.")
    print("*** require_validated will REFUSE to export it. That is the guard")
    print("*** working, not a bug. Narrow the projection, widen the training")
    print("*** sample, or raise the tolerance DELIBERATELY and say so.")

In [ ]:
# COLAB: RUN THIS CELL
# One validation report per projected surface. A projected LST surface sits on top
# of a projected land cover, so it carries BOTH the regression metrics and the
# land-cover Kappa - which is exactly what CLAUDE.md caveat 3 asks for.
PROJECTION_REPORTS = {}
for _key, _surface in SURFACES.items():
    _name, _year = _key
    _scenario = prediction.resolve_scenario(_name, params)
    _row = HORIZONS[HORIZONS["effective_year"] == _year]
    _asked = int(_row["requested_year"].iloc[0]) if not _row.empty else _year
    _notes = [
        f"Land cover projected by CA-Markov from {_base_early}-{_base_late}; LST "
        f"by a random forest fitted on the {EPOCH} cross-section (space-for-time "
        "substitution - the model has never seen time).",
        f"Scenario: {_scenario['label']}"
        + (f", {_scenario['canopy_increase_fraction']:.0%} canopy increase in the "
           f"top {params['prediction']['priority_zones']['top_n']} GN divisions."
           if _scenario["canopy_increase_fraction"] else "."),
        "Priority zones are a PHASE-5 INTERIM proxy ranked on outcome, not "
        "feasibility; Phase 7's MCDA replaces them.",
    ]
    if _asked != _year:
        _notes.append(
            f"Horizon {_year} is the nearest whole Markov step to the requested "
            f"{_asked}; a fractional matrix power is not a valid transition matrix."
        )
    PROJECTION_REPORTS[_key] = prediction.build_validation_report(
        "lst_projection",
        {"rmse": HELD_OUT["rmse"], "r2": HELD_OUT["r2"],
         "kappa": LULC_METRICS["kappa"],
         "persistence_kappa": LULC_METRICS["persistence_kappa"],
         "figure_of_merit": LULC_METRICS["figure_of_merit"]},
        params,
        held_out=True,
        n_train=FITTED["n_train"], n_test=HELD_OUT["n"],
        n_blocks=int(np.unique(SAMPLE["block_id"].to_numpy()[TEST_MASK]).size),
        block_size_m=SPLIT["block_size_m"],
        extrapolation=EXTRAPOLATION[_key],
        notes=_notes,
    )

_refused = {}
for _key, _report in PROJECTION_REPORTS.items():
    try:
        prediction.require_validated(_report, params)
    except prediction.ValidationMissing as _error:
        _refused[_key] = str(_error)
if _refused:
    print(f"*** {len(_refused)} projected surface(s) FAILED the validation guard:")
    for _key, _message in _refused.items():
        print(f"  {_key}: {' '.join(_message.split())[:200]}")
    print("*** They will not be plotted or exported. Fix the cause; do not")
    print("*** bypass the guard.")
else:
    print(f"PASS: all {len(PROJECTION_REPORTS)} projected surfaces are validated.")

# Figures, one horizon at a time, both scenarios on ONE shared colour scale.
for _year in sorted({y for _, y in SURFACES}):
    _panels = {
        prediction.resolve_scenario(n, params)["label"] + f", {_year}":
            SURFACES[(n, _year)]
        for n in SCENARIOS if (n, _year) in SURFACES and (n, _year) not in _refused
    }
    if len(_panels) < 1:
        continue
    _report = PROJECTION_REPORTS[(SCENARIOS[0], _year)]
    _fig = viz.plot_projected_lst(
        _panels, f"figures/projected_lst_{_year}.png", params, _report,
        title=f"Projected dry-season LST, {_year} ({MODEL_SCALE} m)",
    )
    print("Wrote", _fig)
    display(Image(filename=str(_fig)))

    if ("greening", _year) in SURFACES and ("business_as_usual", _year) in SURFACES:
        _difference = SURFACES[("greening", _year)] - SURFACES[("business_as_usual", _year)]
        _fig = viz.plot_scenario_difference(
            _difference, f"figures/scenario_difference_{_year}.png", params, _report,
            title=f"Greening minus business as usual, {_year} ({MODEL_SCALE} m)",
        )
        print("Wrote", _fig)
        display(Image(filename=str(_fig)))
        _finite = _difference[np.isfinite(_difference)]
        print(f"  Mean change over the district: {_finite.mean():+.3f} degC")
        print(f"  Inside priority zones: "
              f"{_difference[PRIORITY_MASK & np.isfinite(_difference)].mean():+.3f} degC")
        print("  A difference of two projections carries BOTH projections'")
        print("  uncertainty. It looks clean because the two surfaces share a")
        print("  model and so share its errors - a property of the method, not")
        print("  evidence that the difference is precise.")

In [ ]:
# COLAB: RUN THIS CELL
# Per-division summary, so the report can name divisions rather than pixels.
_rows = []
for (_name, _year), _surface in SURFACES.items():
    if (_name, _year) in _refused:
        continue
    _means = _zone_mean(_surface)
    _rows.append(pd.DataFrame({
        "zone_id": ZONES["zone_id"].to_numpy(),
        "scenario": _name, "year": _year, "projected_lst_c": _means,
    }))
ZONE_PROJECTION = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
if not ZONE_PROJECTION.empty:
    ZONE_PROJECTION = ZONE_PROJECTION.merge(
        PRIORITY[["zone_id", "rank", "priority"]], on="zone_id", how="left"
    )
    ZONE_PROJECTION.to_csv(
        "data/outputs/projected_lst_by_gn.csv", index=False
    )
    print("Wrote data/outputs/projected_lst_by_gn.csv")
    _wide = ZONE_PROJECTION.pivot_table(
        index=["zone_id", "priority"], columns=["scenario", "year"],
        values="projected_lst_c",
    )
    display(_wide.head(10))
    print()
    print("CAVEAT (caveats.zonal_not_pixel): these are means over administrative")
    print("units. A division-level number is a property of that aggregation and")
    print("must not be read as what any individual location experiences.")
else:
    print("No validated surfaces to summarise by division.")

---
# WAIT HERE (optional) - the MOLUSCE round trip

Part 2 is complete and Phase 6 has a full, validated result. **Part 3 is optional.**

If you want the independent cellular-automaton cross-check, run MOLUSCE in QGIS now.
`docs/molusce_handoff.md` has the whole procedure: which files to load, which tab settings
to use, which outputs to bring back, and what to name them. The short version:

1. QGIS 3.32+, install **MOLUSCE** from the plugin manager (it is actively maintained -
   5.3.0 as of April 2026);
2. mask each `label` band to `observed == 1` before loading;
3. calibrate 2018 -> 2021, simulate one iteration to 2024, validate against observed 2024;
4. recalibrate 2018 -> 2024, simulate one and two iterations (2030 and **2036**);
5. **check MOLUSCE's transition matrix against
   `data/outputs/transition_probabilities_2018_2021.csv`** - if they disagree, stop, because
   the two tools are reading different pixels;
6. bring the results back into `data/interim/` with the names the handoff document lists.

If you are skipping MOLUSCE, run Part 3 anyway - it says so and moves on to the guarded
exports.

---
# PART 3 - reconcile, then export under the guard

In [ ]:
# COLAB: RUN THIS CELL
# Score MOLUSCE's output with the SAME functions as the Python CA, or say it is absent.
_VAL_TARGET = PRIMARY[2]
MOLUSCE_FILES = {
    "validation": f"data/interim/molusce_projected_{_VAL_TARGET}.tif",
    **{f"projection_{y}": f"data/interim/molusce_projected_{y}.tif"
       for y in sorted(PROJECTED)},
}
MOLUSCE = {}
for _label, _path in MOLUSCE_FILES.items():
    print(f"  {_label:18s}: {'OK     ' if os.path.exists(_path) else 'absent '} {_path}")

if not any(os.path.exists(p) for p in MOLUSCE_FILES.values()):
    print()
    print("No MOLUSCE outputs found - the QGIS round trip was skipped.")
    print("That is a supported outcome, not a failure: the Python CA-Markov")
    print("result is fully validated and stands on its own. Record in")
    print("PROGRESS.md that the cross-check was not run, and continue.")
    MOLUSCE_COMPARISON = pd.DataFrame()
else:
    import rasterio

    _rows = []
    for _label, _path in MOLUSCE_FILES.items():
        if not os.path.exists(_path):
            continue
        with rasterio.open(_path) as _handle:
            _labels = _handle.read(1)
        if _labels.shape != VALID.shape:
            raise ValueError(
                f"{_path} is {_labels.shape} but the Earth Engine rasters are "
                f"{VALID.shape}. A misaligned comparison would produce a "
                "confusion matrix full of spurious 'change'. Re-export from "
                "QGIS on EPSG:32644 at "
                f"{CA_SCALE} m over the same extent."
            )
        MOLUSCE[_label] = _labels
        if _label == "validation":
            # Scored through the SAME function, on the SAME mask, with the same
            # class list as the Python CA. Anything else and the comparison is
            # between two metrics rather than two models.
            _rows.append({
                "implementation": "MOLUSCE", "target": _VAL_TARGET,
                **prediction.validate_projection(
                    VALIDATION[PRIMARY]["recode"](LULC[PRIMARY[1]]),
                    VALIDATION[PRIMARY]["recode"](LULC[_VAL_TARGET]),
                    VALIDATION[PRIMARY]["recode"](_labels),
                    params, PROJECTION_CLASSES,
                    mask=VALIDATION[PRIMARY]["mask"],
                ),
            })
    _rows.append({
        "implementation": "Python CA", "target": _VAL_TARGET, **LULC_METRICS,
    })
    MOLUSCE_COMPARISON = pd.DataFrame(_rows)
    MOLUSCE_COMPARISON.to_csv(
        "data/outputs/molusce_vs_python_ca.csv", index=False
    )
    display(MOLUSCE_COMPARISON[[
        "implementation", "target", "n_scored", "kappa", "persistence_kappa",
        "kappa_above_null", "figure_of_merit", "quantity_disagreement",
        "allocation_disagreement",
    ]].round(4))
    print()
    print("Read kappa_above_null first for BOTH. The no-change baseline is the")
    print("same for each, so whichever implementation is closer to it has")
    print("located less of the change.")

    for _year in sorted(PROJECTED):
        _key = f"projection_{_year}"
        if _key not in MOLUSCE:
            continue
        _agree = float(
            (MOLUSCE[_key][VALID] == PROJECTED[_year]["labels"][VALID]).mean()
        )
        print(f"{_year}: the two implementations agree on {_agree:.1%} of cells.")
    print()
    print("A disagreement is a FINDING. MOLUSCE's transition potential is an ANN")
    print("over the driver rasters; the Python CA uses a neighbourhood filter.")
    print("They are not meant to be identical. Do NOT reconcile them by editing")
    print("one until it matches the other - report both and say where they differ.")

## Step 14 - the guard, exercised deliberately

Before anything is written, prove the refusal works on this runtime, with this config.

`prediction.require_validated` is the mechanism CLAUDE.md caveat 3 asks for: a predictive
product physically cannot leave this notebook without computed validation metrics. The
cell below hands it an empty report and expects to be refused. **If it is not refused, stop
and fix the guard** - every export below it is then unprotected.

In [ ]:
# COLAB: RUN THIS CELL
_checks = [
    ("no report at all", None),
    ("metrics missing the land-cover Kappa",
     prediction.build_validation_report(
         "lst_projection", {"rmse": 1.0, "r2": 0.7}, params, held_out=True)),
    ("metrics computed on the TRAINING set",
     prediction.build_validation_report(
         "lst_fit", {"rmse": 1.0, "r2": 0.7}, params, held_out=False)),
    ("too much extrapolation",
     prediction.build_validation_report(
         "lst_fit", {"rmse": 1.0, "r2": 0.7}, params, held_out=True,
         extrapolation={"fraction": 0.9, "tolerance": 0.05,
                        "within_tolerance": False})),
]
_leaked = []
for _label, _report in _checks:
    try:
        prediction.export_projection(None, params, _report, None)
        _leaked.append(_label)
        print(f"  *** NOT REFUSED: {_label}")
    except prediction.ValidationMissing as _error:
        print(f"  refused ({_label}): {' '.join(str(_error).split())[:90]}...")
if _leaked:
    raise RuntimeError(
        f"the validation guard did NOT refuse {_leaked}. Every export below "
        "this cell is unprotected. Stop and fix prediction.require_validated."
    )
print()
print("PASS: the guard refuses every inadequate report. Exports may proceed.")

In [ ]:
# COLAB: RUN THIS CELL
# Write the projected land-cover rasters (guarded) and, if Earth Engine is live,
# submit the server-side projected LST surface (also guarded).
WRITTEN = []
for _year, _result in PROJECTED.items():
    _path = prediction.write_lulc_projection(
        np.where(VALID, _result["labels"], 0), LULC_PROFILE,
        f"data/outputs/lulc_projected_{_year}.tif", params, LULC_REPORT,
    )
    WRITTEN.append(str(_path))
for (_name, _year), _labels in SCENARIO_LABELS.items():
    if (_name, _year) in _refused:
        continue
    _path = prediction.write_lulc_projection(
        np.where(VALID, _labels, 0), LULC_PROFILE,
        f"data/outputs/lulc_projected_{_name}_{_year}.tif", params, LULC_REPORT,
    )
    WRITTEN.append(str(_path))
print(f"Wrote {len(WRITTEN)} projected land-cover raster(s):")
for _path in WRITTEN:
    print("  ", _path)

# Every validation report, in one machine-readable file, so a product on disk can
# always be traced back to the numbers that let it be written.
with open("data/outputs/validation_reports.json", "w", encoding="utf-8") as _fh:
    json.dump(
        {
            "lst_fit": FIT_REPORT,
            "lulc_projection": LULC_REPORT,
            "lst_projection": {f"{n}_{y}": r for (n, y), r in
                               PROJECTION_REPORTS.items()},
        },
        _fh, indent=2, default=float,
    )
print()
print("Wrote data/outputs/validation_reports.json")
print()
print("Caption stamped on every predictive figure:")
print(prediction.validation_caption(
    PROJECTION_REPORTS[(SCENARIOS[0], sorted(PROJECTED)[0])], params
))

In [ ]:
# COLAB: RUN THIS CELL  (needs Earth Engine - skip if this session has none)
# The server-side projected surface, for a full-resolution district raster.
# export_projection calls require_validated BEFORE any ee.batch call, so there is
# no path from an unvalidated model to a written product.
EXPORT_TASKS = []
try:
    _stack = prediction.prediction_stack(params, region=WORK_REGION)
    _sample = prediction.training_sample_collection(params, region=WORK_REGION)
    _ee_rf = prediction.fit_ee_rf(_sample, params)
    _ee_surface = prediction.project_lst_image(_stack, _ee_rf, params)
    _task = prediction.export_projection(
        _ee_surface, params, FIT_REPORT, WORK_REGION,
        product="lst_fitted_surface", scenario=EPOCH,
    )
    EXPORT_TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

    _explained = prediction.ee_rf_explain(_ee_rf)
    if "importance" in _explained:
        CROSS_CHECK = prediction.compare_rf_implementations(
            _explained["importance"], IMPORTANCE
        )
        CROSS_CHECK.to_csv("data/outputs/rf_importance_cross_check.csv", index=False)
        display(CROSS_CHECK)
        print("Agreement on ORDERING is the useful signal. The two importances are")
        print("on different scales (impurity vs permutation) and must never be")
        print("differenced.")
    else:
        print("explain() carried no 'importance'; cross-check skipped.")
except NameError:
    print("No Earth Engine session in this runtime (Part 2 was run standalone).")
    print("Skipping the server-side export. The numpy projection above is the")
    print("validated product; this export is a full-resolution convenience.")
except ee.EEException as _error:
    print("Earth Engine refused the export:", _error)
    print("Report the message; a failed export is a bug to fix.")

if EXPORT_TASKS:
    display(exports.describe_tasks(EXPORT_TASKS))

In [ ]:
# COLAB: RUN THIS CELL
# BRING THE RESULTS HOME.
import zipfile

_bundle = "/content/phase6_outputs.zip"
_written = []
with zipfile.ZipFile(_bundle, "w", zipfile.ZIP_DEFLATED) as _zip:
    for _folder in ("data/outputs", "figures"):
        for _root, _, _files in os.walk(_folder):
            for _name in _files:
                if _name == ".gitkeep":
                    continue
                _path = os.path.join(_root, _name)
                _zip.write(_path, _path)
                _written.append(_path)

print(f"{len(_written)} file(s), {os.path.getsize(_bundle) / 1e6:.1f} MB")
for _path in sorted(_written):
    print("  ", _path)
try:
    from google.colab import files
    files.download(_bundle)
except Exception as _error:
    print(f"(auto-download unavailable: {_error} - use the Files pane)")

## What to check before signing Phase 6 off

### Must pass

1. **Step 0** prints `PASS: all Phase 6 functions and params are present.`, the horizon
   table, **and `PASS: work region ... within 20 % of the expected 699 km2`**. That last
   line is the guard added after run 1, which used an 18,090 km² compositing bounding box
   as the analysis unit and had to be discarded entirely. If it raises, nothing below it
   is worth running.
2. **Step 1** prints the `explain()` keys and the projected band name. Record both in
   `PROGRESS.md` - they are the answer to S1, and every later product depends on them.
3. **Step 2**: every year in `LULC_YEARS` clears
   `prediction.ca_markov.min_observed_fraction`. **2021 has never been probed before**; if
   it fails, the calibration triplet moves and Steps 9-12 must be re-run.
4. Every Part 1 task reads `COMPLETED` in the poll cell.
5. **Step 7** reports blocked cross-validation with a fold-to-fold range, and a held-out
   RMSE and R2.
6. **Step 8** reports the random-vs-blocked R2 gap. Quote this number in the write-up.
7. **Step 9** reports Kappa, the no-change-null Kappa, the Pontius split and the figure
   of merit **for both class schemes**, on land only, with the excluded-cell count. Record
   which scheme the projection used, and why, as **S9**.
   * Run 1 scored Kappa 0.928 against a null of 0.933 — **worse than doing nothing** — with
     a figure of merit of 0.019. If that repeats under both schemes, the CA has
     demonstrated no skill at locating change and the projection must say so.
8. **Step 10** reports the GHSL cross-check as a **cell-count** comparison against a
   thresholded built-dominant mask, with the fraction-weighted area beside it and labelled
   as the different quantity it is.
9. **Step 14** prints `PASS: the guard refuses every inadequate report.` If it does not,
   stop.
10. `pytest tests/ -q` **inside Colab**. Locally this is **1029 passed, 14 skipped**; the
    skips are `esda`, `libpysal`, `spreg`, `mgwr`, `pymannkendall`, `rasterio`,
    `geopandas` and `sklearn` cross-checks that DO resolve in Colab, so **expect fewer
    skips and more passes there** - report the exact counts.

### Judgement calls to report, not to fix silently

* **A metric below its floor.** `min_kappa` and `min_figure_of_merit` are floors, not
  targets. A CA-Markov model tuned until it clears its own validation set has no
  validation set. Report the failure and what it means for the projection.
* **The random split not scoring higher** in Step 8. Run 1 produced a gap of only
  +0.009 R², but over an 18,090 km² region where a smooth regional gradient dominated the
  signal — the blocked split was never actually stressed. Over 699 km² the gap should be
  informative. If it is still near zero, that is now a real finding about the block size,
  and a larger `split.block_size_m` goes in as a sensitivity.
* **Which class scheme the projection used**, and by how much the two differed. If
  grouping is what makes the CA clear its floors, the report must say that the result
  depends on collapsing trees, grass and shrub into one class — and that the greening
  lever is correspondingly weaker.
* **Thin classes** in Step 12's substitution table. They are painted anyway, because a
  projected cell needs some value - but name them.
* **Allocation shortfall** in Step 10. It means the Markov demand exceeded the changeable
  donor cells. It is a column, not an error.
* **MOLUSCE disagreeing with the Python CA.** That is a finding. Its transition potential
  is an ANN over the driver rasters; ours is a neighbourhood filter. Report both.

### Known limits that travel into Phase 7

1. **Nothing here is a forecast.** Every product is conditional on the calibrated
   transition rates continuing and the fitted LST-driver relationship holding.
2. **The forest has never seen time.** Space-for-time substitution, and it cannot
   extrapolate; Step 12 quantifies how far outside its envelope each scenario goes.
3. **Priority zones are an interim proxy** ranked on outcome, not feasibility.
   `prediction.priority_zones.source` is `phase5_interim`; Phase 7's MCDA/AHP replaces it,
   and `apply_greening_scenario` takes a plain zone list so nothing else changes.
4. **Population is WorldPop 2020**, held constant, because that is where the dataset ends.
5. **Land cover is Dynamic World at 100 m**, aggregated by mode from 10 m. Phase 5 measured
   that no absolute green figure is classifier-independent (WorldCover 2021 green 71.2 %
   vs Dynamic World 2024 44.6 %). Quote the classifier and the scale with every area.
6. **The greening scenario's placement rule is an assumption**, not a finding: converted
   cells are chosen by neighbourhood canopy share, which favours contiguous planting.
7. **A scenario difference carries both projections' uncertainty.** It is the least certain
   product in this phase, not the most.